# Char LM
이 노트북에서는 가장 기초적인 형태의 **decoder-only Transformer**를 구현합니다.

<img src="./img/charLM.png" width="800">

일반적인 서브워드 토큰 대신 각 글자(character)를 하나의 토큰으로 사용하는 character-level 방식을 사용하며, Transformer 기반 언어 모델의 기본 
구조와 학습 과정을 이해하는 것을 목표로 합니다.

### 1. text <-> token

### LLM의 학습 데이터

기본적인 **word-level tokenization**에서의 데이터 구성 방식은 다음과 같다.  
다만 실제 LLM에서는 보통 word-level보다 **subword tokenizer**를 사용한다.

예를 들어 다음 문장이 있다고 하자.

`나는 사과를 먹었다.`

Tokenizer가 이를 다음과 같이 변환한다고 가정하자.

```text
나는   → 15
사과를 → 83
먹었다 → 233
.      → 3
```

그러면 문장은 다음과 같은 토큰 ID의 나열로 표현할 수 있다.

```python
tokens = [15, 83, 233, 3]
```

LLM의 기본적인 학습 목표는 **이전 토큰들을 바탕으로 다음 토큰을 예측하는 것**이다.

따라서 입력과 정답을 한 칸씩 어긋나게 구성할 수 있다.

```text
입력: [나는, 사과를, 먹었다]
정답: [사과를, 먹었다, .]
```

Causal Mask를 적용하면 하나의 시퀀스 안에서 다음과 같은 예측을 동시에 학습할 수 있다.

```text
나는                 → 사과를
나는 사과를          → 먹었다
나는 사과를 먹었다   → .
```

즉, 토큰 시퀀스 `x`와 정답 시퀀스 `y`를 한 칸 차이 나게 구성하는 것만으로 **next-token prediction**을 위한 학습 데이터를 만들 수 있다.

한편 **character-level tokenization**에서는 단어 대신 문자 하나를 하나의 토큰으로 사용한다. 예를 들어 `사과`는 `사`, `과`라는 두 토큰으로 나뉠 수 있으며, 각각의 토큰은 모델 내부에서 하나의 정수형 **token ID**로 표현된다.

* input.txt는 한국어 위키백과의 인공지능 관련 문서에서 추출한 텍스트 3만자로로 구성되어 있습니다.

In [3]:
with open("input.txt", "r", encoding = "utf-8") as f:
    text = f.read()

print(text[:100])

인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 


### 전처리

위키백과의 위키문법을 제거하는 전처리를 진행한다.

In [13]:
import re

def preprocess(text):
    # 위키 문법 제거
    text = re.sub(r'=+', '', text)

    # HTML 공백
    text = text.replace('&#x20;', ' ')

    # 공백, 탭, 줄바꿈 ->  공백 1개
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

text = preprocess(text)

print(text[:100])
print("문자 수:", len(text))


인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 
문자 수: 29375


### 토크나이저 만들기